# Gold Benchmarks And Retrieval Comparisons

This notebook shows benchmark fixtures for spans, triples, and relations; deterministic graph-store scale fixtures; graph train/validation splits; node2vec and KG embeddings; scalable PageRank/community backends; GraphSAGE preparation; and retrieval comparisons across text-only, vector, and graph-aware signals.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            ROOT = parent
            break
sys.path.insert(0, str(ROOT / "src"))


## Evaluate against gold spans, triples, and relations

Gold fixtures make evaluation explicit instead of relying only on proxy metrics.


In [ ]:
from kg_construction.application.graph_store_benchmark import generate_graph_store_benchmark_fixture
from kg_construction.domain.graph_objects import PropertyEdge, PropertyGraph, PropertyNode
from kg_construction.domain.models import ConstructionBatch, EntityCandidate, EvidenceRef, GraphFact, RelationCandidate, SourceRecord
from kg_construction.infrastructure.enrichment import (
    GraphRetrievalBenchmark,
    GraphSageLinkPredictionTrainer,
    GraphTrainValidationSplitter,
    KnowledgeGraphEmbeddingAdapter,
    LargeGraphAnalyticsBackend,
    Node2VecEmbeddingAdapter,
    PyTorchGeometricGraphSAGEAdapter,
)
from kg_construction.infrastructure.evaluation import GoldBenchmarkEvaluator, GoldBenchmarkFixture
from kg_construction.infrastructure.loaders.stubs import AdapterNotConfigured

evidence_source = SourceRecord.from_text("INCIDENT-1042 affected Recommendation API.", source_name="memory://gold")
evidence = EvidenceRef.build(evidence_source, text="INCIDENT-1042")
kg_batch = ConstructionBatch(
    source_records=[evidence_source],
    entities=[
        EntityCandidate.build(entity_type="Incident", name="INCIDENT-1042", evidence=[evidence], extractor_name="notebook", technique="gold"),
        EntityCandidate.build(entity_type="System", name="Recommendation API", evidence=[evidence], extractor_name="notebook", technique="gold"),
        EntityCandidate.build(entity_type="Date", name="2026-07-09", evidence=[evidence], extractor_name="notebook", technique="gold"),
    ],
    facts=[
        GraphFact.build(subject="urn:kg:incident:INCIDENT-1042", predicate="http://www.w3.org/1999/02/22-rdf-syntax-ns#type", object="urn:kg:schema:Incident", evidence=[evidence], extractor_name="notebook", technique="gold"),
        GraphFact.build(subject="urn:kg:incident:INCIDENT-1042", predicate="urn:kg:schema:affectedSystem", object="urn:kg:system:recommendation-api", evidence=[evidence], extractor_name="notebook", technique="gold"),
    ],
    relations=[
        RelationCandidate.build(source="entity_incident_1042", relation_type="AFFECTED", target="entity_recommendation_api", evidence=[evidence], extractor_name="notebook", technique="gold"),
        RelationCandidate.build(source="entity_incident_1042", relation_type="INVESTIGATED_BY", target="entity_platform_team", evidence=[evidence], extractor_name="notebook", technique="gold"),
    ],
)

evaluator = GoldBenchmarkEvaluator()
ner_gold = GoldBenchmarkFixture.from_json_file(ROOT / "tests" / "fixtures" / "gold_standards" / "ner_spans.json")
triple_gold = GoldBenchmarkFixture.from_json_file(ROOT / "tests" / "fixtures" / "gold_standards" / "gold_triples.json")
relation_gold = GoldBenchmarkFixture.from_json_file(ROOT / "tests" / "fixtures" / "gold_standards" / "relation_sample.json")
{
    "entities": evaluator.evaluate_entities(kg_batch, ner_gold).model_dump(mode="json"),
    "triples": evaluator.evaluate_triples(kg_batch, triple_gold).model_dump(mode="json"),
    "relations": evaluator.evaluate_relations(kg_batch, relation_gold).model_dump(mode="json"),
}


## Graph ML preparation and embedding adapters

The graph ML layer can split edges, add node2vec-style embeddings, add KG embeddings, run large-graph PageRank/community backends, and prepare GraphSAGE/link-prediction payloads.


In [ ]:
graph = generate_graph_store_benchmark_fixture(node_count=12, edge_factor=2)
split = GraphTrainValidationSplitter().split(graph)
graph = Node2VecEmbeddingAdapter(dimensions=4).enrich(graph)
graph = KnowledgeGraphEmbeddingAdapter(dimensions=4).enrich(graph)
graph = LargeGraphAnalyticsBackend().pagerank(graph)
graph = LargeGraphAnalyticsBackend().community_detection(graph)
trainer_payload = GraphSageLinkPredictionTrainer().prepare(graph)

try:
    pyg_embeddings = PyTorchGeometricGraphSAGEAdapter().encode(graph)
    pyg_status = f"encoded {len(pyg_embeddings)} nodes"
except AdapterNotConfigured as exc:
    pyg_status = str(exc)

{
    "validation_edges": len(split.validation_edges),
    "negative_validation_edges": len(split.negative_validation_edges),
    "node2vec_metadata": graph.metadata["node2vec"],
    "kg_embedding_metadata": graph.metadata["knowledge_graph_embeddings"],
    "large_graph_pagerank_backend": graph.metadata["large_graph_pagerank_backend"],
    "graphsage_payload_keys": sorted(trainer_payload),
    "pyg_adapter_status": pyg_status,
}


## Retrieval comparison: text-only vs vector vs graph-aware

The retrieval benchmark compares signal availability and latency tradeoffs for text-only RAG, vector RAG, and GraphRAG-style graph context.


In [ ]:
retrieval_graph = graph.model_copy(
    update={
        "nodes": [
            node.model_copy(
                update={
                    "properties": {
                        **node.properties,
                        "evidence_texts": [node.properties.get("name", node.node_id)] if index < 4 else [],
                    }
                }
            )
            for index, node in enumerate(graph.nodes)
        ]
    }
)
rows = GraphRetrievalBenchmark().compare(retrieval_graph, k=3)
[row.model_dump(mode="json") for row in rows]


## Batch-size fixture for store benchmarking

Synthetic graph fixtures give deterministic high-volume inputs for graph-store batch sizing and ingestion tests.


In [ ]:
large_fixture = generate_graph_store_benchmark_fixture(node_count=100, edge_factor=3)
{
    "nodes": len(large_fixture.nodes),
    "edges": len(large_fixture.edges),
    "fixture": large_fixture.metadata["fixture"],
}
